In [9]:
# run before: aws configure

In [1]:
!pwd

/workspace/LIDAR/sonata_ws/sonata-workspace/data/data_exploration


# Imports

In [1]:
# !pip install tqdm
# !python -m pip install pyarrow
# !python -m pip install awscli
# !python -m pip install laspy
# !python -m pip install py3dtiles
# !python -m pip install s3fs


In [ ]:
# run before: aws configure

In [1]:
from pathlib import Path
import shutil
import subprocess
import tempfile
import sys

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import laspy
from tqdm.auto import tqdm

sys.path.append('..')
from map_from_scans import load_poses



/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. Prepare file structure

# Code

In [14]:
!touch .keep

In [15]:
!aws s3 cp .keep s3://3d-scenes/datasets/semantic_kitti/scenes/00/sequences/00/sensors/lidar.velodyne/frames/.keep \
  --endpoint-url https://storage.yandexcloud.net

!aws s3 cp .keep s3://3d-scenes/datasets/semantic_kitti/scenes/00/maps/lidar_pointcloud/.keep \
  --endpoint-url https://storage.yandexcloud.net

upload: ./.keep to s3://3d-scenes/datasets/semantic_kitti/scenes/00/sequences/00/sensors/lidar.velodyne/frames/.keep
upload: ./.keep to s3://3d-scenes/datasets/semantic_kitti/scenes/00/maps/lidar_pointcloud/.keep


# Checks

In [12]:
!aws s3 ls s3://3d-scenes/raw/semantic_kitti/sequences/ \
  --endpoint-url https://storage.yandexcloud.net

                           PRE 00/
                           PRE 01/
                           PRE 02/
                           PRE 03/
                           PRE 04/


In [1]:
!aws s3 ls s3://3d-scenes/datasets/semantic_kitti/scenes/00/ \
  --recursive \
  --endpoint-url https://storage.yandexcloud.net

2026-03-09 15:54:50          0 datasets/semantic_kitti/scenes/00/maps/lidar_pointcloud/.keep
2026-03-09 15:54:49          0 datasets/semantic_kitti/scenes/00/sequences/00/sensors/lidar.velodyne/frames/.keep


In [23]:
!pwd

/workspace/LIDAR/sonata_ws/sonata-workspace/data/data_exploration


# 2. Move sequences data 

# Code

In [31]:
S3_ENDPOINT = "https://storage.yandexcloud.net"
MOVING_MIN = 252
MOVING_MAX = 259
MIN_DISTANCE = 3.5


def aws_s3(*args: str, capture_output: bool = True) -> str:
    result = subprocess.run(
        ["aws", "s3", *args, "--no-progress", "--only-show-errors", "--endpoint-url", S3_ENDPOINT],
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        capture_output=capture_output,
    )
    return result.stdout if capture_output else ""


def list_remote_sequences(remote_root: str) -> list[str]:
    out = aws_s3("ls", remote_root)
    seqs = []
    for line in out.splitlines():
        parts = line.split()
        if len(parts) == 2 and parts[1].endswith("/"):
            name = parts[1].strip("/")
            if name.isdigit():
                seqs.append(name)
    return sorted(seqs)

def load_times(path: Path) -> np.ndarray:
    return np.loadtxt(path, dtype=np.float64)


def decode_labels(path: Path) -> tuple[np.ndarray | None, np.ndarray | None]:
    if not path.exists():
        return None, None
    labels = np.fromfile(path, dtype=np.uint32)
    semantic = (labels & 0xFFFF).astype(np.int32)
    instance = (labels >> 16).astype(np.int32)
    return semantic, instance


def write_poses_parquet(poses: list[np.ndarray], times: np.ndarray, out_path: Path) -> None:
    n = len(poses)
    table = pa.table({
        "frame_id": np.array([f"{i:06d}" for i in range(n)], dtype=object),
        "timestamp": times,
        "m00": np.array([p[0, 0] for p in poses], dtype=np.float32),
        "m01": np.array([p[0, 1] for p in poses], dtype=np.float32),
        "m02": np.array([p[0, 2] for p in poses], dtype=np.float32),
        "m03": np.array([p[0, 3] for p in poses], dtype=np.float32),
        "m10": np.array([p[1, 0] for p in poses], dtype=np.float32),
        "m11": np.array([p[1, 1] for p in poses], dtype=np.float32),
        "m12": np.array([p[1, 2] for p in poses], dtype=np.float32),
        "m13": np.array([p[1, 3] for p in poses], dtype=np.float32),
        "m20": np.array([p[2, 0] for p in poses], dtype=np.float32),
        "m21": np.array([p[2, 1] for p in poses], dtype=np.float32),
        "m22": np.array([p[2, 2] for p in poses], dtype=np.float32),
        "m23": np.array([p[2, 3] for p in poses], dtype=np.float32),
    })
    out_path.parent.mkdir(parents=True, exist_ok=True)
    pq.write_table(table, out_path, compression="zstd")


def make_frame_table(points, semantic, instance, is_static, frame_id):
    n = len(points)

    data = {
        "frame_id": np.full(n, frame_id, dtype=object),
        "x": points[:, 0].astype(np.float32),
        "y": points[:, 1].astype(np.float32),
        "z": points[:, 2].astype(np.float32),
        "intensity": points[:, 3].astype(np.float32),
        "is_static": is_static.astype(bool),
    }

    if semantic is not None:
        data["semantic_id"] = semantic.astype(np.int32)

    if instance is not None:
        data["instance_id"] = instance.astype(np.int32)

    return pa.table(data)


def concat_tables(tables):
    if not tables:
        return None
    return pa.concat_tables(tables, promote_options="default")


def flush_batch(tables, out_path: Path, remote_path: str):
    table = concat_tables(tables)
    if table is None:
        return
    pq.write_table(table, out_path, compression="zstd")
    upload_file(out_path, remote_path)
    out_path.unlink(missing_ok=True)




def make_gt_table(
    xyz: np.ndarray,
    intensity: np.ndarray,
    frame_id: str,
    timestamp: float,
    semantic: np.ndarray | None,
    instance: np.ndarray | None,
) -> pa.Table:
    n = len(xyz)
    data = {
        "x": xyz[:, 0].astype(np.float32),
        "y": xyz[:, 1].astype(np.float32),
        "z": xyz[:, 2].astype(np.float32),
        "intensity": intensity.astype(np.float32),
        "frame_id": np.full(n, frame_id, dtype=object),
        "timestamp": np.full(n, timestamp, dtype=np.float64),
    }
    if semantic is not None:
        data["semantic_id"] = semantic
    if instance is not None:
        data["instance_id"] = instance
    return pa.table(data)


def filter_static(
    points: np.ndarray,
    semantic: np.ndarray | None,
    instance: np.ndarray | None,
) -> tuple[np.ndarray, np.ndarray | None, np.ndarray | None]:
    if semantic is not None:
        mask = (semantic < MOVING_MIN) | (semantic > MOVING_MAX)
        points = points[mask]
        semantic = semantic[mask]
        if instance is not None:
            instance = instance[mask]

    dist = np.linalg.norm(points[:, :3], axis=1)
    mask = dist > MIN_DISTANCE
    points = points[mask]
    if semantic is not None:
        semantic = semantic[mask]
    if instance is not None:
        instance = instance[mask]

    return points, semantic, instance


def transform_points(points: np.ndarray, pose: np.ndarray) -> np.ndarray:
    ones = np.ones((len(points), 1), dtype=np.float32)
    pts_h = np.hstack([points[:, :3], ones])
    return (pose @ pts_h.T).T[:, :3]


def upload_file(local_path: Path, remote_path: str) -> None:
    aws_s3("cp", str(local_path), remote_path, capture_output=False)


def download_sequence(remote_path: str, local_path: Path) -> None:
    local_path.mkdir(parents=True, exist_ok=True)
    aws_s3("sync", remote_path, str(local_path), capture_output=False)

def compute_is_static(
    points: np.ndarray,
    semantic: np.ndarray | None,
) -> np.ndarray:
    is_static = np.ones(len(points), dtype=bool)

    if semantic is not None:
        is_static &= (semantic < MOVING_MIN) | (semantic > MOVING_MAX)

    dist = np.linalg.norm(points[:, :3], axis=1)
    is_static &= dist > MIN_DISTANCE

    return is_static

In [32]:
def transfer_all_frames(
    local_dataset_root: str,
    bucket_root: str,
    sequences=None,
    batch_size: int = 100,
):
    local_root = Path(local_dataset_root).resolve()

    local_raw_root = local_root / "sequences"
    remote_raw_root = f"{bucket_root}/raw/semantic_kitti/sequences"
    remote_dataset_root = f"{bucket_root}/datasets/semantic_kitti/scenes"

    if sequences is None:
        local_sequences = sorted(
            p.name for p in local_raw_root.glob("*")
            if p.is_dir() and p.name.isdigit()
        )
        sequences = local_sequences or list_remote_sequences(remote_raw_root)

    if not sequences:
        raise RuntimeError("No sequences found")

    for seq in sequences:
        print(f"Sequence {seq}")

        downloaded = False
        local_seq = local_raw_root / seq

        if not local_seq.exists():
            download_sequence(f"{remote_raw_root}/{seq}", local_seq)
            downloaded = True

        try:
            velodyne_dir = local_seq / "velodyne"
            labels_dir = local_seq / "labels"

            calib_path = local_seq / "calib.txt"
            poses_path = local_seq / "poses.txt"
            times_path = local_seq / "times.txt"

            poses = load_poses(str(calib_path), str(poses_path))
            times = load_times(times_path)
            frame_files = sorted(velodyne_dir.glob("*.bin"))

            if len(frame_files) != len(poses) or len(frame_files) != len(times):
                raise ValueError(
                    f"Length mismatch in sequence {seq}: "
                    f"frames={len(frame_files)}, poses={len(poses)}, times={len(times)}"
                )

            scene_id = seq
            sequence_id = "00"

            remote_frames_root = (
                f"{remote_dataset_root}/{scene_id}/sequences/{sequence_id}"
                "/sensors/lidar.velodyne/batches"
            )
            remote_map_root = f"{remote_dataset_root}/{scene_id}/maps/lidar_pointcloud/batches"

            with tempfile.TemporaryDirectory(prefix="skitti_") as tmp_dir:
                tmp_root = Path(tmp_dir)

                frames_batch_tables = []
                maps_batch_tables = []
                batch_idx = 0

                frames_batch_tmp = tmp_root / "frames_batch.parquet"
                maps_batch_tmp = tmp_root / "maps_batch.parquet"

                bar = tqdm(
                    frame_files,
                    total=len(frame_files),
                    desc=f"Frames {seq}",
                    unit="frame",
                    dynamic_ncols=True,
                    miniters=1,
                    mininterval=0.2,
                )

                for i, frame_path in enumerate(bar):
                    frame_id = frame_path.stem
                    pose = poses[i]

                    points = np.fromfile(frame_path, dtype=np.float32).reshape(-1, 4)
                    semantic, instance = decode_labels(labels_dir / f"{frame_id}.label")
                    is_static = compute_is_static(points, semantic)

                    frame_table = make_frame_table(
                        points=points,
                        semantic=semantic,
                        instance=instance,
                        is_static=is_static,
                        frame_id=frame_id,
                    )
                    frames_batch_tables.append(frame_table)

                    xyz_world = transform_points(points, pose)
                    gt_points = np.empty_like(points)
                    gt_points[:, :3] = xyz_world
                    gt_points[:, 3] = points[:, 3]

                    gt_table = make_frame_table(
                        points=gt_points,
                        semantic=semantic,
                        instance=instance,
                        is_static=is_static,
                        frame_id=frame_id,
                    )
                    maps_batch_tables.append(gt_table)

                    should_flush = (
                        len(frames_batch_tables) >= batch_size
                        or i == len(frame_files) - 1
                    )

                    if should_flush:
                        batch_name = f"{batch_idx:06d}.parquet"

                        flush_batch(
                            frames_batch_tables,
                            frames_batch_tmp,
                            f"{remote_frames_root}/{batch_name}",
                        )
                        flush_batch(
                            maps_batch_tables,
                            maps_batch_tmp,
                            f"{remote_map_root}/{batch_name}",
                        )

                        frames_batch_tables.clear()
                        maps_batch_tables.clear()
                        batch_idx += 1

                    bar.set_postfix(frame=frame_id, batch=batch_idx)

        finally:
            if downloaded and local_seq.exists():
                shutil.rmtree(local_seq, ignore_errors=True)

# Execution

In [33]:
transfer_all_frames(
    local_dataset_root="/workspace/datasets/semantic_kitti",
    bucket_root="s3://3d-scenes",
    # sequences=["00", "01", "02", "03", "04"],
    sequences=["01"],
)

Sequence 01


Frames 01: 100%|██████████| 1101/1101 [04:18<00:00,  4.25frame/s, batch=12, frame=001100]


# 3. Points to Viewer

# Code

In [39]:
!py3dtiles

/usr/bin/sh: 1: py3dtiles: not found


# Execution

In [2]:
from utils import build_viewer_tiles

build_viewer_tiles(
    local_root=".",
    bucket_root="s3://3d-scenes",
    scene_id="01",
)

01/lidar_pointcloud: 100%|██████████| 12/12 [01:51<00:00,  9.26s/batch]


Point tiler - summary:
  - files to process: [PosixPath('/workspace/LIDAR/sonata_ws/sonata-workspace/data/data_exploration/viewer_tiles/01/lidar_pointcloud/las/000000.las'), PosixPath('/workspace/LIDAR/sonata_ws/sonata-workspace/data/data_exploration/viewer_tiles/01/lidar_pointcloud/las/000001.las'), PosixPath('/workspace/LIDAR/sonata_ws/sonata-workspace/data/data_exploration/viewer_tiles/01/lidar_pointcloud/las/000002.las'), PosixPath('/workspace/LIDAR/sonata_ws/sonata-workspace/data/data_exploration/viewer_tiles/01/lidar_pointcloud/las/000003.las'), PosixPath('/workspace/LIDAR/sonata_ws/sonata-workspace/data/data_exploration/viewer_tiles/01/lidar_pointcloud/las/000004.las'), PosixPath('/workspace/LIDAR/sonata_ws/sonata-workspace/data/data_exploration/viewer_tiles/01/lidar_pointcloud/las/000005.las'), PosixPath('/workspace/LIDAR/sonata_ws/sonata-workspace/data/data_exploration/viewer_tiles/01/lidar_pointcloud/las/000006.las'), PosixPath('/workspace/LIDAR/sonata_ws/sonata-workspace/dat

{'scene_id': '01',
 'map_id': 'lidar_pointcloud',
 'batch_count': 12,
 'las_count': 12,
 'local_tiles': PosixPath('/workspace/LIDAR/sonata_ws/sonata-workspace/data/data_exploration/viewer_tiles/01/lidar_pointcloud/tiles'),
 'remote_tiles': 's3://3d-scenes/datasets/semantic_kitti/scenes/01/maps/lidar_pointcloud/tiles'}

In [4]:
print('finish')

finish


# 3. Library testing 

In [9]:
!mkdir /workspace/datasets/semantic_kitti_cache

In [9]:
!ls /workspace/datasets/semantic_kitti/sequences/01

calib.txt  image_1  image_3  poses.txt	velodyne
image_0    image_2  labels   times.txt


In [6]:
!ls /workspace/datasets/semantic_kitti_cache

cache  e9b6c0198997b6736fc9b3cc29f90fcf1196da8828ee0f206c57918c6b737e2b


In [10]:
!aws s3 ls s3://3d-scenes/datasets/semantic_kitti/scenes/01/sequences/00/lidar.velodyne/batches/000011.parquet \
  --endpoint-url https://storage.yandexcloud.net

2026-03-10 19:22:37    1987964 000011.parquet


In [5]:
import fsspec
import pyarrow.parquet as pq

S3_ENDPOINT = "https://storage.yandexcloud.net"
fs = fsspec.filesystem(
    "filecache",
    target_protocol="s3",
    target_options={
        "endpoint_url": S3_ENDPOINT,
    },
    cache_storage="/workspace/datasets/semantic_kitti_cache",
)

with fs.open(
    "3d-scenes/datasets/semantic_kitti/scenes/01/sequences/02/lidar.velodyne/batches/000011.parquet",
    "rb",
) as f:
    todo = pq.ParquetFile(f).metadata.num_rows
    print(todo)


FileNotFoundError: 3d-scenes/datasets/semantic_kitti/scenes/01/sequences/02/lidar.velodyne/batches/000011.parquet